# 4주차 — GPT구조와_프롬프트_비교

- 100분 제출형 실습 · 가설 → 구현/실험 → 증거 → 해석
- 학생용 TODO 완성 후 다음 셀 실행 · TODO에서 중단되는 것은 의도된 동작임
- 환경·모델 로딩·반복 학습은 지원 코드로 제공
- 답안은 개조식으로 기록 · 결과 방향보다 비교 조건과 해석을 평가
- Colab: 런타임 → 런타임 유형 변경 → GPU 권장
- 4주차 GPT-3 원형 API 이외의 기본 실습은 API 키 불필요


In [20]:
%pip -q install "transformers==4.57.6" "pandas>=2.2,<3" "matplotlib>=3.8,<3.11" "ftfy>=6.2,<7"

In [21]:
import os, gc, re
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed


SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))
print("Device:", DEVICE)

def cleanup():
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()


Device: cuda


## 문제 4-1. GPT-1 방식 Post-LN에서 warmup이 정말 필요한가?

- 질문: GPT-1의 Post-LN과 GPT-2의 Pre-LN을 코드와 통제 실험으로 구분할 수 있는가?
- 구현: 아래 작은 블록의 두 forward 경로 완성
- 실험: 폭·깊이·공유 가중치·입력·정답 고정, LN 위치만 변경, 3개 seed 비교
- 제출: 구조 식별 근거, 완성 코드, 층별 gradient 그래프
- 식: Pre-LN $x+F(LN(x))$, Post-LN $LN(x+F(x))$
- 초기 gradient 실험은 GPT-1/2 전체 사전학습 재현이 아님
- 추가 실험: Post-LN/Pre-LN × warmup 없음/있음 × 최대 학습률 2개 × seed 3개
- GPT-1 방식의 작은 무작위 초기화 causal LM 사용 · 학습 완료된 GPT-1의 추가 학습과 구분

In [ ]:
import inspect, copy
from transformers.models.openai.modeling_openai import Block as GPT1Block
from transformers.models.gpt2.modeling_gpt2 import GPT2Block
for name,cls in [("GPT-1",GPT1Block),("GPT-2",GPT2Block)]:
    source=inspect.getsource(cls.forward)
    print("\n",name,"forward:\n",source)

In [ ]:
class NormBlock(nn.Module):
    def __init__(self,pre=True,d=32):
        super().__init__(); self.pre=pre
        self.ln1=nn.LayerNorm(d); self.ln2=nn.LayerNorm(d)
        self.attn=nn.MultiheadAttention(d,4,dropout=0.,batch_first=True)
        self.ffn=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d))
    def attention(self,x):
        mask=torch.triu(torch.ones(x.shape[1],x.shape[1],dtype=torch.bool,device=x.device),1)
        return self.attn(x,x,x,attn_mask=mask,need_weights=False)[0]
    def forward(self,x):
        if self.pre:
            raise NotImplementedError("TODO: Pre-LN attention·FFN 잔차 경로 구현 필요")
        else:
            raise NotImplementedError("TODO: Post-LN attention·FFN 잔차 경로 구현 필요")

In [ ]:
for pre in [True,False]:
    block=NormBlock(pre); probe=torch.randn(2,8,32,requires_grad=True)
    result=block(probe)
    assert result.shape==probe.shape and torch.isfinite(result).all()
    result.square().mean().backward()
    assert probe.grad is not None and torch.isfinite(probe.grad).all()
DEPTH=8
gradient_rows=[]
for seed in [0,1,2]:
    torch.manual_seed(seed)
    pre_blocks=nn.ModuleList([NormBlock(True) for _ in range(DEPTH)])
    post_blocks=copy.deepcopy(pre_blocks)
    for block in post_blocks: block.pre=False
    x=torch.randn(2,8,32); target=torch.randn(2,8,32)
    final_norm=nn.LayerNorm(32); head=nn.Linear(32,32)
    for label,blocks in [("pre",pre_blocks),("post",post_blocks)]:
        head.zero_grad(); final_norm.zero_grad(); h=x
        for block in blocks: h=block(h)
        loss=F.mse_loss(head(final_norm(h)),target); loss.backward()
        for i,block in enumerate(blocks):
            gradient_rows.append({"seed":seed,"norm":label,"layer":i+1,
                                  "gradient_norm":float(block.attn.in_proj_weight.grad.norm())})
gradient_table=pd.DataFrame(gradient_rows)
for label,group in gradient_table.groupby("norm"):
    agg=group.groupby("layer").gradient_norm.agg(["mean","std"])
    plt.errorbar(agg.index,agg["mean"],yerr=agg["std"],label=label,capsize=3)
plt.xlabel("Layer"); plt.ylabel("Attention gradient norm"); plt.legend(); plt.show()

### 추가 실험 — Post-LN에서 warmup의 필요성 검증

- 가설: 동일 최대 학습률에서 warmup이 초기 손실 급증·기울기 크기에 미치는 영향
- 연구 근거: [Xiong et al., 2020](https://proceedings.mlr.press/v119/xiong20b.html)의 LN 위치·초기 gradient 분석
- 범위: GPT-1 방식 Post-LN 블록을 갖는 축소 모델의 처음부터 학습 · 원형 GPT-1 재현 아님
- 통제: 동일 초기 가중치·데이터·미니배치 순서·optimizer·step 수·최대 학습률
- 추가 통제: 두 구조 모두 최종 LN 사용, dropout·gradient clipping·weight decay 미사용
- 학습률: 0.001, 0.01을 실행 전 고정 · seed 0/1/2 · 40-step · warmup 10-step
- 과제: 작은 합성 토큰 수열의 next-token prediction · 실제 자연어 corpus와의 차이 명시
- TODO: 첫 update부터 선형 증가하는 학습률 함수 완성
- 제출: learning rate·train loss·gradient 그래프, seed별 결과표, Post-LN에서 warmup 효과 해석
- 판단 기준: 비유한 값 발생 여부, 초기 10-step 최대 손실, 최대 gradient norm, 최종 validation loss
- warmup은 초기 누적 학습률도 줄임 · 결과가 좋아도 원인을 LN만으로 설명 불가



In [ ]:
def learning_rate_at(step, peak_lr, warmup_steps):
    # step: 0부터 시작 · warmup=0이면 항상 peak_lr
    # warmup=10이면 첫 update 0.1배, 열 번째 update부터 1배
    raise NotImplementedError("TODO: 선형 warmup 후 일정 학습률 구현 필요")

In [ ]:
assert learning_rate_at(0,1.,0)==1.
assert abs(learning_rate_at(0,1.,10)-0.1)<1e-8
assert learning_rate_at(9,1.,10)==learning_rate_at(20,1.,10)==1.
WARMUP_CONFIG={"steps":40,"warmup_steps":10,"peak_lrs":[0.001,0.01],
               "seeds":[0,1,2],"depth":6,"width":32,"batch":8,"length":16,"vocab":32}
class TinyWarmupLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(32,32); self.pos=nn.Parameter(torch.zeros(1,16,32))
        self.blocks=nn.ModuleList([NormBlock(True) for _ in range(6)])
        self.final_ln=nn.LayerNorm(32); self.head=nn.Linear(32,32)
    def forward(self,ids):
        h=self.embed(ids)+self.pos[:,:ids.shape[1]]
        for block in self.blocks: h=block(h)
        return self.head(self.final_ln(h))

generator=torch.Generator().manual_seed(2026)
def make_sequences(n):
    start=torch.randint(0,32,(n,1),generator=generator)
    stride=torch.randint(1,5,(n,1),generator=generator)
    return (start+stride*torch.arange(17))%32
train_sequences=make_sequences(256)
validation_sequences=make_sequences(64).to(DEVICE)
batch_indices=torch.randint(0,256,(40,8),generator=generator)
warmup_rows=[]; warmup_summaries=[]
for seed in WARMUP_CONFIG["seeds"]:
    torch.manual_seed(seed)
    template=TinyWarmupLM()
    for peak_lr in WARMUP_CONFIG["peak_lrs"]:
        for norm in ["post","pre"]:
            for warmup_steps in [0,10]:
                model=copy.deepcopy(template).to(DEVICE)
                for block in model.blocks: block.pre=(norm=="pre")
                optimizer=torch.optim.AdamW(model.parameters(),lr=peak_lr,weight_decay=0.)
                run=[]; status="finite"
                for step in range(40):
                    model.train(); optimizer.zero_grad(set_to_none=True)
                    batch=train_sequences[batch_indices[step]].to(DEVICE)
                    lr=learning_rate_at(step,peak_lr,warmup_steps)
                    for group in optimizer.param_groups: group["lr"]=lr
                    loss=F.cross_entropy(model(batch[:,:-1]).reshape(-1,32),batch[:,1:].reshape(-1))
                    loss.backward()
                    grad=torch.sqrt(sum(p.grad.detach().float().square().sum() for p in model.parameters() if p.grad is not None))
                    row={"seed":seed,"peak_lr":peak_lr,"norm":norm,"warmup_steps":warmup_steps,
                         "step":step+1,"lr":lr,"loss":float(loss.detach()),"gradient_norm":float(grad)}
                    run.append(row); warmup_rows.append(row)
                    if not (torch.isfinite(loss) and torch.isfinite(grad)):
                        status="nonfinite_loss_or_gradient"; break
                    optimizer.step()
                model.eval()
                with torch.no_grad():
                    val=F.cross_entropy(model(validation_sequences[:,:-1]).reshape(-1,32),validation_sequences[:,1:].reshape(-1))
                if not torch.isfinite(val): status="nonfinite_validation"
                warmup_summaries.append({"seed":seed,"peak_lr":peak_lr,"norm":norm,"warmup_steps":warmup_steps,
                    "status":status,"completed_steps":len(run),"first_loss":run[0]["loss"],
                    "early_peak_loss":max(r["loss"] for r in run[:10]),
                    "loss_spike_over_2x":any(r["loss"]>2*run[0]["loss"] for r in run),
                    "max_gradient_norm":max(r["gradient_norm"] for r in run),
                    "final_validation_loss":float(val),"lr_sum":sum(r["lr"] for r in run)})
                del model,optimizer
warmup_curves=pd.DataFrame(warmup_rows); warmup_summary=pd.DataFrame(warmup_summaries)
display(warmup_summary)
display(warmup_summary.groupby(["peak_lr","norm","warmup_steps"])[
    ["early_peak_loss","max_gradient_norm","final_validation_loss","lr_sum"]].agg(["mean","std"]))
fig,axes=plt.subplots(2,3,figsize=(15,7))
for row_index,peak_lr in enumerate(WARMUP_CONFIG["peak_lrs"]):
    subset=warmup_curves[warmup_curves.peak_lr==peak_lr]
    for (norm,warm),group in subset.groupby(["norm","warmup_steps"]):
        for col,metric in enumerate(["lr","loss","gradient_norm"]):
            agg=group.groupby("step")[metric].agg(["mean","std"])
            ax=axes[row_index,col]
            ax.plot(agg.index,agg["mean"],label=f"{norm}, warmup={warm}")
            ax.fill_between(agg.index,agg["mean"]-agg["std"].fillna(0),agg["mean"]+agg["std"].fillna(0),alpha=.1)
            ax.set_title(f"peak lr={peak_lr}: {metric}"); ax.set_xlabel("Update")
            if metric=="gradient_norm": ax.set_yscale("log")
            ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 문제 4-2. GPT-1·GPT-2는 같은 과제를 얼마나 수행하는가?
- 실행: GPT-1·GPT-2 × 3과제 × 4문항 × 2프롬프트 = 48회
- 통제: 같은 입력·예시·greedy 생성·최대 24개 새 토큰
- tokenizer가 다르므로 같은 토큰 수가 같은 문자 길이를 뜻하지 않음
- 영어 base 모델에 chat template 사용하지 않음
- 성공/실패 원문 확인, 오류 유형 직접 분류
- 프롬프트 개선 이후 동일 문항 3개 재평가
- 제출: 결과표, 오류 사례, 개선 전후 출력, 일반화의 한계

In [22]:
TASKS={
 "sentiment":{"instruction":"Classify the sentiment. Output only positive or negative.",
  "demos":[("I enjoyed every minute of the movie.","positive"),("The meal was terrible.","negative")],
  "eval":[("The service was wonderful.","positive"),("I regret buying this broken product.","negative"),
          ("The book made me happy.","positive"),("The trip was disappointing.","negative")]},
 "extraction":{"instruction":"Extract the person's name. Output only the name.",
  "demos":[("Alice bought a blue notebook.","Alice"),("Bob arrived at noon.","Bob")],
  "eval":[("Mary found a key.","Mary"),("John visited a museum.","John"),
          ("Emma wrote a letter.","Emma"),("David lost his umbrella.","David")]},
 "format":{"instruction":"Join the words with a comma and no spaces. Output only the result.",
  "demos":[("red blue","red,blue"),("cat dog","cat,dog")],
  "eval":[("sun moon","sun,moon"),("tea coffee","tea,coffee"),
          ("pen book","pen,book"),("day night","day,night")]}
}

In [23]:
def score_output(task,text,answer):
    # 정답 단어 포함 점수와 엄격 일치 점수를 분리 · 최종 판정은 원문과 대조
    if task=="sentiment": match=re.search(r"\b(positive|negative)\b",text.lower())
    elif task=="extraction": match=re.search(r"\b"+re.escape(answer)+r"\b",text,re.I)
    else: match=re.search(re.escape(answer),text,re.I)
    content_correct=bool(match) and (task!="sentiment" or match.group(1)==answer)
    return {"content_correct":content_correct,"strict_match":text.strip().lower()==answer.lower()}

In [24]:
def build_prompt(task,text,shots):
    spec=TASKS[task]
    prefix=spec["instruction"]+"\n\n"
    for x,y in spec["demos"][:shots]: prefix+=f"Input: {x}\nOutput: {y}\n\n"
    return prefix+f"Input: {text}\nOutput:"

In [25]:
for task,spec in TASKS.items():
    assert len(spec["eval"])==4 and len(spec["demos"])>=2
    assert not ({x for x,y in spec["eval"]}&{x for x,y in spec["demos"]})
    for shots in [0,2]:
        prompt=build_prompt(task,spec["eval"][0][0],shots)
        assert isinstance(prompt,str) and spec["eval"][0][0] in prompt
        assert prompt.count("Input:")==shots+1
        print(task,shots,"shot\n",prompt,"\n")

sentiment 0 shot
 Classify the sentiment. Output only positive or negative.

Input: The service was wonderful.
Output: 

sentiment 2 shot
 Classify the sentiment. Output only positive or negative.

Input: I enjoyed every minute of the movie.
Output: positive

Input: The meal was terrible.
Output: negative

Input: The service was wonderful.
Output: 

extraction 0 shot
 Extract the person's name. Output only the name.

Input: Mary found a key.
Output: 

extraction 2 shot
 Extract the person's name. Output only the name.

Input: Alice bought a blue notebook.
Output: Alice

Input: Bob arrived at noon.
Output: Bob

Input: Mary found a key.
Output: 

format 0 shot
 Join the words with a comma and no spaces. Output only the result.

Input: sun moon
Output: 

format 2 shot
 Join the words with a comma and no spaces. Output only the result.

Input: red blue
Output: red,blue

Input: cat dog
Output: cat,dog

Input: sun moon
Output: 



In [26]:
MODELS=[("GPT-1","openai-community/openai-gpt","1e0d4f3028acbffb47fe933cea64619c5ec1a002"),
        ("GPT-2","openai-community/gpt2","607a30d783dfa663caf39e06633721c8d4cfcd7e")]
MAX_NEW=24
local_rows=[]
for label,model_id,revision in MODELS:
    tok=AutoTokenizer.from_pretrained(model_id,revision=revision)
    net=AutoModelForCausalLM.from_pretrained(model_id,revision=revision,use_safetensors=True).to(DEVICE).eval()
    pad_id=tok.eos_token_id if tok.eos_token_id is not None else tok.unk_token_id
    for task,spec in TASKS.items():
        for index,(text,answer) in enumerate(spec["eval"]):
            for shots in [0,2]:
                prompt=build_prompt(task,text,shots)
                inp=tok(prompt,return_tensors="pt").to(DEVICE)
                assert inp.input_ids.shape[1]+MAX_NEW<=net.config.n_positions
                with torch.no_grad():
                    generated=net.generate(**inp,max_new_tokens=MAX_NEW,do_sample=False,pad_token_id=pad_id)
                raw=tok.decode(generated[0,inp.input_ids.shape[1]:],skip_special_tokens=True)
                local_rows.append({"model":label,"model_id":model_id,"revision":revision,"task":task,
                  "item":index,"shots":shots,"prompt":prompt,"expected":answer,"raw":raw,
                  "input_tokens":inp.input_ids.shape[1],**score_output(task,raw,answer)})
    del net; cleanup()
local_table=pd.DataFrame(local_rows)
display(local_table.groupby(["model","task","shots"])[["content_correct","strict_match"]].mean())
display(local_table[["model","task","shots","expected","raw"]].head(12))

content_correct  strict_match
model task       shots                               
GPT-1 extraction 0                 1.00           0.0
                 2                 0.00           0.0
      format     0                 0.00           0.0
                 2                 0.00           0.0
      sentiment  0                 0.00           0.0
                 2                 0.00           0.0
GPT-2 extraction 0                 1.00           0.0
                 2                 1.00           0.0
      format     0                 0.00           0.0
                 2                 0.75           0.0
      sentiment  0                 0.00           0.0
                 2                 0.50           0.0

,model,task,shots,expected,raw
0,GPT-1,sentiment,0,positive,"the service was great . \n "" i 'm sorry , "" i ..."
1,GPT-1,sentiment,2,positive,i enjoyed every minute of the movie . \n i was...
2,GPT-1,sentiment,0,negative,i regret buying this broken product . \n i 'm ...
3,GPT-1,sentiment,2,negative,i regret buying this broken product . \n i was...
4,GPT-1,sentiment,0,positive,the book made me happy . \n i was n't sure wha...
5,GPT-1,sentiment,2,positive,i enjoyed every minute of the movie . \n i was...
6,GPT-1,sentiment,0,negative,"the trip was disappointing . \n "" i 'm sorry ,..."
7,GPT-1,sentiment,2,negative,i enjoyed the movie . \n i was n't sure what t...
8,GPT-1,extraction,0,Mary,mary found a key . output : mary found a key ....
9,GPT-1,extraction,2,Mary,alice found a key . output : alice found a key...


###아래 improved_prompt 부분을 개선하라

In [27]:
def improved_prompt(task,text):
    pairs=TASKS[task]["demos"]
    return "\n".join(f"{x} => {y}" for x,y in pairs)+f"\n{text} =>"

In [28]:
tok=AutoTokenizer.from_pretrained(MODELS[1][1],revision=MODELS[1][2])
net=AutoModelForCausalLM.from_pretrained(MODELS[1][1],revision=MODELS[1][2],use_safetensors=True).to(DEVICE).eval()
improved=[]
for task,spec in TASKS.items():
    text,answer=spec["eval"][0]; prompt=improved_prompt(task,text)
    inp=tok(prompt,return_tensors="pt").to(DEVICE)
    assert inp.input_ids.shape[1]+MAX_NEW<=net.config.n_positions
    with torch.no_grad(): generated=net.generate(**inp,max_new_tokens=MAX_NEW,do_sample=False,pad_token_id=tok.eos_token_id)
    raw=tok.decode(generated[0,inp.input_ids.shape[1]:],skip_special_tokens=True)
    improved.append({"task":task,"prompt":prompt,"raw":raw,"expected":answer,**score_output(task,raw,answer)})
display(pd.DataFrame(improved))
del net; cleanup()

,task,prompt,raw,expected,content_correct,strict_match
0,sentiment,I enjoyed every minute of the movie. => positi...,positive\nThe food was good. => negative\nThe...,positive,True,False
1,extraction,Alice bought a blue notebook. => Alice\nBob ar...,Mary\nAlice found a key. => Mary\nAlice found...,Mary,True,False
2,format,"red blue => red,blue\ncat dog => cat,dog\nsun ...","sun,moon,sun,sun,sun,sun,sun,sun,sun,sun,sun,...","sun,moon",True,False


In [29]:
# 비교 대상: GPT-2, 기존 2-shot, 과제별 첫 번째 문항
before = local_table[
    (local_table["model"] == "GPT-2")
    & (local_table["shots"] == 2)
    & (local_table["item"] == 0)
][["task", "raw", "content_correct", "strict_match"]]

after = pd.DataFrame(improved)[
    ["task", "expected", "raw", "content_correct", "strict_match"]
]

comparison = before.merge(
    after,
    on="task",
    suffixes=("_before", "_after"),
    validate="one_to_one",
)
assert len(comparison) == len(TASKS)


def judge(before, after):
    if before == after:
        return "변화 없음"
    return "개선" if after else "악화"


for metric in ["content_correct", "strict_match"]:
    comparison[f"{metric}_판정"] = [
        judge(b, a)
        for b, a in zip(
            comparison[f"{metric}_before"],
            comparison[f"{metric}_after"],
        )
    ]

# 원문과 문항별 판정
with pd.option_context("display.max_colwidth", None):
    display(comparison)

# 지표별 결과 요약
n = len(comparison)
print(f"평가 범위: GPT-2, 동일한 {n}개 문항, 기존 2-shot 대비\n")

for metric, label in [
    ("content_correct", "내용 점수"),
    ("strict_match", "엄격 일치"),
]:
    before_count = int(comparison[f"{metric}_before"].sum())
    after_count = int(comparison[f"{metric}_after"].sum())
    judgments = comparison[f"{metric}_판정"]

    print(
        f"{label}: {before_count}/{n} → {after_count}/{n}"
        f" | 개선 {(judgments == '개선').sum()}문항,"
        f" 악화 {(judgments == '악화').sum()}문항,"
        f" 변화 없음 {(judgments == '변화 없음').sum()}문항"
    )


print(
    "\n해석 주의: 이 결과는 선택한 3문항에 한정됩니다. "
    "내용 점수는 문자열 기반이므로 생성 원문과 대조해야 합니다. "
    "여러 프롬프트 요소를 함께 바꿨으므로 개별 변경의 효과는 분리할 수 없습니다."
)

,task,raw_before,content_correct_before,strict_match_before,expected,raw_after,content_correct_after,strict_match_after,content_correct_판정,strict_match_판정
0,sentiment,positive\n\nInput: The movie was great.\n\nOutput: positive\n\nInput: The movie was great.,True,False,positive,positive\nThe food was good. => negative\nThe movie was great. => positive\nThe movie was great. =>,True,False,변화 없음,변화 없음
1,extraction,Mary\n\nInput: Alice\n\nInput: Mary\n\nInput: Mary\n\nInput: Mary\n\nInput,True,False,Mary,Mary\nAlice found a key. => Mary\nAlice found a key. => Mary\nAlice found a key. =>,True,False,변화 없음,변화 없음
2,format,"sun,moon\n\nOutput: sun,moon\n\nOutput: sun,moon\n\nOutput: sun,moon",True,False,"sun,moon","sun,moon,sun,sun,sun,sun,sun,sun,sun,sun,sun,sun,",True,False,변화 없음,변화 없음


평가 범위: GPT-2, 동일한 3개 문항, 기존 2-shot 대비

내용 점수: 3/3 → 3/3 | 개선 0문항, 악화 0문항, 변화 없음 3문항
엄격 일치: 0/3 → 0/3 | 개선 0문항, 악화 0문항, 변화 없음 3문항

해석 주의: 이 결과는 선택한 3문항에 한정됩니다. 내용 점수는 문자열 기반이므로 생성 원문과 대조해야 합니다. 여러 프롬프트 요소를 함께 바꿨으므로 개별 변경의 효과는 분리할 수 없습니다.
